In [0]:
%sql
CREATE VOLUME IF NOT EXISTS smart_telemetry_electricity_usage_pricing.default.silver_checkpoints;

CREATE TABLE IF NOT EXISTS smart_telemetry_electricity_usage_pricing.default.silver_meter_telemetry (
  meter_id          STRING     NOT NULL,
  tariff_grid_id    STRING,
  reading_ts        TIMESTAMP  NOT NULL,
  kwh_consumed      DECIMAL(18,6),
  voltage           DOUBLE,
  event_hub         STRING,
  partition         INT,
  offset            BIGINT,
  enqueued_time     TIMESTAMP,
  silver_loaded_at  TIMESTAMP
);

CREATE TABLE IF NOT EXISTS smart_telemetry_electricity_usage_pricing.default.silver_meter_telemetry_quarantine (
  body            STRING,
  reason          STRING,
  event_hub       STRING,
  partition       INT,
  offset          BIGINT,
  enqueued_time   TIMESTAMP,
  quarantined_at  TIMESTAMP
);

In [0]:
from pyspark.sql.functions import col, from_json, current_timestamp, when
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, DecimalType

SCHEMA = "smart_telemetry_electricity_usage_pricing.default"
CHECKPOINT = "/Volumes/smart_telemetry_electricity_usage_pricing/default/silver_checkpoints/silver_meter_telemetry"

meter_schema = StructType([
    StructField("meter_id",       StringType()),
    StructField("tariff_grid_id", StringType()),
    StructField("kwh_consumed",   DecimalType(18, 6)),   # exact, read straight from the JSON text
    StructField("voltage",        DoubleType()),
    StructField("timestamp",      TimestampType()),
])

parsed = (spark.readStream.table(f"{SCHEMA}.bronze_meter_telemetry")
    .withColumn("j", from_json(col("body"), meter_schema))
    .select("body", "event_hub", "partition", "offset", "enqueued_time",
            col("j.meter_id").alias("meter_id"),
            col("j.tariff_grid_id").alias("tariff_grid_id"),
            col("j.timestamp").alias("reading_ts"),
            col("j.kwh_consumed").alias("kwh_consumed"),
            col("j.voltage").alias("voltage")))

def load_batch(batch_df, batch_id):
    session = batch_df.sparkSession          # the batch's own session, not the notebook's `spark`

    reason = (when(col("meter_id").isNull(), "missing meter_id")
              .when(col("reading_ts").isNull(), "missing or invalid timestamp")
              .when(col("kwh_consumed").isNull(), "missing or invalid kwh_consumed")
              .when(col("kwh_consumed") < 0, "negative kwh_consumed"))
    checked = batch_df.withColumn("reason", reason)

    # 1. bad rows: keep them, with the reason
    (checked.filter(col("reason").isNotNull())
        .select("body", "reason", "event_hub", "partition", "offset", "enqueued_time")
        .withColumn("quarantined_at", current_timestamp())
        .write.mode("append").saveAsTable(f"{SCHEMA}.silver_meter_telemetry_quarantine"))

    # 2. good rows: de-duplicate by business key, insert only keys not already in silver
    good = (checked.filter(col("reason").isNull())
            .dropDuplicates(["meter_id", "reading_ts"])
            .drop("body", "reason")
            .withColumn("silver_loaded_at", current_timestamp()))
    good.createOrReplaceTempView("silver_meter_batch")
    session.sql(f"""
        MERGE INTO {SCHEMA}.silver_meter_telemetry AS t
        USING silver_meter_batch AS s
        ON t.meter_id = s.meter_id AND t.reading_ts = s.reading_ts
        WHEN NOT MATCHED THEN INSERT *
    """)

q = (parsed.writeStream
     .foreachBatch(load_batch)
     .option("checkpointLocation", CHECKPOINT)
     .trigger(availableNow=True)
     .start())
q.awaitTermination()

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM smart_telemetry_electricity_usage_pricing.default.bronze_meter_telemetry)            AS bronze_rows,
  (SELECT COUNT(*) FROM smart_telemetry_electricity_usage_pricing.default.silver_meter_telemetry)            AS silver_rows,
  (SELECT COUNT(*) FROM smart_telemetry_electricity_usage_pricing.default.silver_meter_telemetry_quarantine) AS quarantined_rows,
  (SELECT COUNT(*) FROM (SELECT meter_id, reading_ts FROM smart_telemetry_electricity_usage_pricing.default.silver_meter_telemetry
                         GROUP BY meter_id, reading_ts HAVING COUNT(*) > 1))                                AS duplicate_keys

bronze_rows,silver_rows,quarantined_rows,duplicate_keys
450000,450000,0,0


In [0]:
%sql
CREATE TABLE IF NOT EXISTS smart_telemetry_electricity_usage_pricing.default.silver_day_ahead_pricing (
  bidding_zone      STRING     NOT NULL,
  interval_start    TIMESTAMP  NOT NULL,
  price             DECIMAL(12,4),
  price_unit        STRING,
  currency          STRING,
  event_hub         STRING,
  partition         INT,
  offset            BIGINT,
  enqueued_time     TIMESTAMP,
  silver_loaded_at  TIMESTAMP
);

CREATE TABLE IF NOT EXISTS smart_telemetry_electricity_usage_pricing.default.silver_day_ahead_pricing_quarantine (
  body            STRING,
  reason          STRING,
  event_hub       STRING,
  partition       INT,
  offset          BIGINT,
  enqueued_time   TIMESTAMP,
  quarantined_at  TIMESTAMP
);

In [0]:
from pyspark.sql.functions import col, from_json, current_timestamp, when, upper, row_number
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DecimalType
from pyspark.sql.window import Window

SCHEMA = "smart_telemetry_electricity_usage_pricing.default"
PRICING_CHECKPOINT = "/Volumes/smart_telemetry_electricity_usage_pricing/default/silver_checkpoints/silver_day_ahead_pricing"

pricing_schema = StructType([
    StructField("timestamp",        TimestampType()),     # start of the 15-minute interval
    StructField("value",            DecimalType(12, 4)),  # exact, read straight from the JSON text
    StructField("price_unit",       StringType()),
    StructField("currency",         StringType()),
    StructField("eic_bidding_zone", StringType()),
])

pricing_parsed = (spark.readStream.table(f"{SCHEMA}.bronze_day_ahead_pricing")
    .withColumn("j", from_json(col("body"), pricing_schema))
    .select("body", "event_hub", "partition", "offset", "enqueued_time",
            col("j.eic_bidding_zone").alias("bidding_zone"),
            col("j.timestamp").alias("interval_start"),
            col("j.value").alias("price"),
            upper(col("j.price_unit")).alias("price_unit"),
            col("j.currency").alias("currency")))

def load_pricing_batch(batch_df, batch_id):
    session = batch_df.sparkSession

    reason = (when(col("bidding_zone").isNull(), "missing bidding_zone")
              .when(col("interval_start").isNull(), "missing or invalid interval_start")
              .when(col("price").isNull(), "missing or invalid price")
              .when(~col("price_unit").isin("MWH", "KWH"), "unknown price_unit"))
    checked = batch_df.withColumn("reason", reason)

    # 1. bad rows: keep them, with the reason
    (checked.filter(col("reason").isNotNull())
        .select("body", "reason", "event_hub", "partition", "offset", "enqueued_time")
        .withColumn("quarantined_at", current_timestamp())
        .write.mode("append").saveAsTable(f"{SCHEMA}.silver_day_ahead_pricing_quarantine"))

    # 2. good rows: newest copy of each (zone, interval) in this batch
    latest = Window.partitionBy("bidding_zone", "interval_start").orderBy(
        col("enqueued_time").desc(), col("offset").desc())
    good = (checked.filter(col("reason").isNull())
            .withColumn("rn", row_number().over(latest))
            .filter(col("rn") == 1)
            .drop("rn", "body", "reason")
            .withColumn("silver_loaded_at", current_timestamp()))
    good.createOrReplaceTempView("silver_pricing_batch")

    # 3. upsert: a newer price for an existing interval replaces the old one
    session.sql(f"""
        MERGE INTO {SCHEMA}.silver_day_ahead_pricing AS t
        USING silver_pricing_batch AS s
        ON t.bidding_zone = s.bidding_zone AND t.interval_start = s.interval_start
        WHEN MATCHED AND s.enqueued_time > t.enqueued_time THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

q = (pricing_parsed.writeStream
     .foreachBatch(load_pricing_batch)
     .option("checkpointLocation", PRICING_CHECKPOINT)
     .trigger(availableNow=True)
     .start())
q.awaitTermination()

In [0]:
%sql
SELECT bidding_zone,
       COUNT(*)            AS intervals,
       MIN(interval_start) AS first_interval,
       MAX(interval_start) AS last_interval,
       MIN(price)          AS min_price,
       MAX(price)          AS max_price,
       (SELECT COUNT(*) FROM smart_telemetry_electricity_usage_pricing.default.silver_day_ahead_pricing_quarantine) AS quarantined
FROM smart_telemetry_electricity_usage_pricing.default.silver_day_ahead_pricing
GROUP BY bidding_zone
ORDER BY bidding_zone

bidding_zone,intervals,first_interval,last_interval,min_price,max_price,quarantined
SE1,96,2026-10-30T23:00:00.000Z,2026-10-31T22:45:00.000Z,21.0900,104.8800,0
SE2,94,2026-10-30T23:00:00.000Z,2026-10-31T22:45:00.000Z,22.2200,125.2400,0
SE3,95,2026-10-30T23:00:00.000Z,2026-10-31T22:45:00.000Z,123.6600,195.7500,0
SE4,96,2026-10-30T23:00:00.000Z,2026-10-31T22:45:00.000Z,210.2000,301.9100,0


In [0]:
%sql
SET TIME ZONE 'UTC';

CREATE OR REPLACE TABLE smart_telemetry_electricity_usage_pricing.default.silver_price_grid AS
WITH p AS (
  SELECT *, to_date(from_utc_timestamp(interval_start, 'Europe/Stockholm')) AS delivery_day
  FROM smart_telemetry_electricity_usage_pricing.default.silver_day_ahead_pricing),
days AS (SELECT DISTINCT bidding_zone, delivery_day FROM p),
expected AS (
  SELECT d.bidding_zone, d.delivery_day,
         explode(sequence(
           to_utc_timestamp(CAST(d.delivery_day AS TIMESTAMP), 'Europe/Stockholm'),
           to_utc_timestamp(CAST(date_add(d.delivery_day, 1) AS TIMESTAMP), 'Europe/Stockholm') - INTERVAL 15 MINUTES,
           INTERVAL 15 MINUTES)) AS interval_start
  FROM days d),
joined AS (
  SELECT e.bidding_zone, e.delivery_day, e.interval_start, p.price, p.price_unit, p.currency
  FROM expected e
  LEFT JOIN p ON p.bidding_zone = e.bidding_zone AND p.interval_start = e.interval_start)
SELECT bidding_zone,
       delivery_day,
       interval_start,
       last_value(price, true)      OVER w AS price,
       last_value(price_unit, true) OVER w AS price_unit,
       last_value(currency, true)   OVER w AS currency,
       price IS NULL                       AS is_filled,
       current_timestamp()                 AS built_at
FROM joined
WINDOW w AS (PARTITION BY bidding_zone, delivery_day ORDER BY interval_start
             ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT bidding_zone, delivery_day,
       COUNT(*)                                  AS intervals,
       SUM(CASE WHEN is_filled THEN 1 ELSE 0 END) AS filled,
       SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END) AS still_unpriced
FROM smart_telemetry_electricity_usage_pricing.default.silver_price_grid
GROUP BY bidding_zone, delivery_day
ORDER BY bidding_zone, delivery_day

bidding_zone,delivery_day,intervals,filled,still_unpriced
SE1,2026-10-31,96,0,0
SE2,2026-10-31,96,2,0
SE3,2026-10-31,96,1,0
SE4,2026-10-31,96,0,0


In [0]:
%sql
SELECT * FROM smart_telemetry_electricity_usage_pricing.default.silver_price_grid
WHERE is_filled ORDER BY bidding_zone, interval_start

bidding_zone,delivery_day,interval_start,price,price_unit,currency,is_filled,built_at
SE2,2026-10-31,2026-10-31T08:15:00.000Z,38.0000,MWH,EUR,true,2026-11-15T01:21:14.735Z
SE2,2026-10-31,2026-10-31T14:30:00.000Z,38.0000,MWH,EUR,true,2026-11-15T01:21:14.735Z
SE3,2026-10-31,2026-10-31T12:00:00.000Z,146.0000,MWH,EUR,true,2026-11-15T01:21:14.735Z


In [0]:
%sql
WITH p AS (SELECT * FROM smart_telemetry_electricity_usage_pricing.default.silver_day_ahead_pricing),
bounds AS (SELECT MIN(interval_start) AS s, MAX(interval_start) AS e FROM p),
expected AS (
  SELECT z.bidding_zone, explode(sequence(b.s, b.e, INTERVAL 15 MINUTES)) AS interval_start
  FROM bounds b CROSS JOIN (SELECT DISTINCT bidding_zone FROM p) z)
SELECT e.bidding_zone, e.interval_start,
       prev.price AS price_15_min_before,
       nxt.price  AS price_15_min_after
FROM expected e
LEFT JOIN p      ON p.bidding_zone = e.bidding_zone AND p.interval_start = e.interval_start
LEFT JOIN p prev ON prev.bidding_zone = e.bidding_zone AND prev.interval_start = e.interval_start - INTERVAL 15 MINUTES
LEFT JOIN p nxt  ON nxt.bidding_zone = e.bidding_zone AND nxt.interval_start = e.interval_start + INTERVAL 15 MINUTES
WHERE p.bidding_zone IS NULL
ORDER BY 1, 2

bidding_zone,interval_start,price_15_min_before,price_15_min_after
SE2,2026-10-31T08:15:00.000Z,38.0000,28.2800
SE2,2026-10-31T14:30:00.000Z,38.0000,34.6300
SE3,2026-10-31T12:00:00.000Z,146.0000,150.6800


In [0]:
%sql
SELECT typeof(kwh_consumed) AS kwh_type, MIN(kwh_consumed) AS min_kwh, MAX(kwh_consumed) AS max_kwh,
       MIN(reading_ts) AS first_reading, MAX(reading_ts) AS last_reading
FROM smart_telemetry_electricity_usage_pricing.default.silver_meter_telemetry

kwh_type,min_kwh,max_kwh,first_reading,last_reading
"decimal(18,6)",0.000403,0.002527,2026-10-30T03:55:22.726Z,2026-10-30T04:44:50.477Z
